# Evaluating a Classification Prompt

A prompt is tuned like a machine learning model: you measure it on a **test set** whose right answers you know, rather than judging it on two or three tries.

Here, an LLM sorts IT support tickets into five categories. We compare several versions of the prompt on 40 hand-labeled tickets.

The model: [Qwen3.5-2B](https://huggingface.co/Qwen/Qwen3.5-2B), a small open model released in 2026. A GPU is recommended (`Runtime > Change runtime type`). Without a GPU, the notebook keeps the same model, but takes about 20 minutes (a rough estimate).

In [1]:
import logging
import warnings

import huggingface_hub
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore", message=r"(?s).*HF_TOKEN.*")
huggingface_hub.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

# Without the optional causal_conv1d and flash-linear-attention packages, transformers
# says it computes Qwen3.5's layers in plain PyTorch: slower, same result.
logging.getLogger("transformers.integrations.hub_kernels").addFilter(
  lambda record: "falling back to its reference PyTorch" not in record.getMessage()
)

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cpu":
  warnings.warn(
    "No GPU found: the notebook keeps the same model, but takes about 20 minutes (a rough estimate). "
    "To go faster, pick a GPU (Runtime > Change runtime type), then rerun the notebook."
  )

LLM = "Qwen/Qwen3.5-2B"
DTYPE = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(LLM, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(LLM, dtype=DTYPE, device_map=device)

## The test set

40 tickets, 8 per category, with the category support would have chosen. Some are ambiguous on purpose: a phishing email is about email, but belongs to security; a VPN password is about the network, but belongs to access.

In [2]:
CATEGORIES = ["access", "network", "email", "workstation", "security"]

TICKETS = [
  ("My account is locked after several wrong passwords.", "access"),
  ("I don't have the rights to open the Finance shared folder.", "access"),
  ("My VPN password has expired, how do I change it?", "access"),
  ("New hire on Monday: please create their account and mailbox.", "access"),
  ("I no longer receive the two-factor authentication codes on my phone.", "access"),
  ("Can I get write access to the team wiki?", "access"),
  ("I forgot my Windows password.", "access"),
  ("Outlook asks for my password again every time it starts.", "access"),
  ("The VPN disconnects every ten minutes.", "network"),
  ("No wifi in meeting room B.", "network"),
  ("The internet has been very slow on the whole floor since this morning.", "network"),
  ("The file server can't be reached from the Lyon branch.", "network"),
  ('The VPN won\'t connect: "gateway unreachable" error.', "network"),
  ("Video calls keep dropping from the Nantes site.", "network"),
  ("The network socket in my office no longer works.", "network"),
  ("The intranet site is down, for everyone.", "network"),
  ("I haven't received any emails since last night.", "email"),
  ("My mailbox is full, I can't send messages anymore.", "email"),
  ("How do I share my calendar with my assistant?", "email"),
  ("A customer says my emails end up in their spam folder.", "email"),
  ("I'd like a mailing list for the project team.", "email"),
  ("A 30 MB attachment is rejected when sending.", "email"),
  ("My out-of-office replies aren't sent.", "email"),
  ("Meeting invitations arrive with a one-hour offset.", "email"),
  ("My screen stays black at startup.", "workstation"),
  ("The 3rd-floor printer prints blank pages.", "workstation"),
  ("I need Excel on my new laptop.", "workstation"),
  ("My computer has been very slow since the last update.", "workstation"),
  ("Several keys on my laptop keyboard don't respond anymore.", "workstation"),
  ("My laptop battery only lasts an hour now.", "workstation"),
  ("The accounting software crashes when it opens.", "workstation"),
  ("My headset isn't recognized by the computer.", "workstation"),
  ('An email from "IT" asks me to confirm my password through a link.', "security"),
  ("I clicked a link in an email and a page asked for my credentials.", "security"),
  ("The antivirus reports an infected file on my USB stick.", "security"),
  ("I lost my work laptop on the train.", "security"),
  ("Pop-up ads open by themselves on my computer.", "security"),
  ("Someone logged into my account from abroad last night.", "security"),
  (
    "A supplier asks by email to change their bank details before the next payment.",
    "security",
  ),
  ("I sent a customer file to an outside address by mistake.", "security"),
]
test_set = pd.DataFrame(TICKETS, columns=["ticket", "expected"])
test_set.expected.value_counts()

,count
expected,
access,8
network,8
email,8
workstation,8
security,8


## Measuring a prompt

The `evaluate` function sends each ticket to the model with the prompt (`{ticket}` is replaced by the ticket's text), reads the first category named in the answer, and computes the share of right answers. An answer that names no category counts as an error, shown as `?`.

In [3]:
def ask(prompts: list[str], batch_size: int = 8) -> list[str]:
  answers = []
  for start in range(0, len(prompts), batch_size):
    messages = [
      [{"role": "user", "content": p}] for p in prompts[start : start + batch_size]
    ]
    inputs = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      enable_thinking=False,
      padding=True,
      return_tensors="pt",
      return_dict=True,
    ).to(device)
    with torch.no_grad():
      output = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
      )
    answers += tokenizer.batch_decode(
      output[:, inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
  return answers


def category(answer: str) -> str:
  answer = answer.lower()
  found = [(answer.find(c), c) for c in CATEGORIES if c in answer]
  return min(found)[1] if found else "?"


def evaluate(prompt: str) -> pd.DataFrame:
  answers = ask([prompt.replace("{ticket}", t) for t in test_set.ticket])
  results = test_set.assign(answer=answers, predicted=[category(a) for a in answers])
  accuracy = (results.predicted == results.expected).mean()
  print(
    f"Accuracy: {accuracy:.0%} ({(results.predicted == results.expected).sum()}/{len(results)})"
  )
  return results


def errors(results: pd.DataFrame) -> pd.DataFrame:
  return results[results.predicted != results.expected][
    ["ticket", "expected", "predicted"]
  ]

## Version 1: the naive prompt

In [4]:
PROMPT_V1 = """Put this ticket in a category: access, network, email, workstation or security.

Ticket: {ticket}"""

v1 = evaluate(PROMPT_V1)
v1.answer.head()

Accuracy: 40% (16/40)


,answer
0,"Based on the description, the issue involves a..."
1,"Based on the description, the issue is related..."
2,"Based on the nature of the request, this ticke..."
3,**access**\n\n**Reasoning:**\nThe ticket reque...
4,"Based on the issue described, the problem is r..."


The model answers with sentences of varying length: easy for a human to read, less so for a program. Which mistakes does it make?

In the run below, 10 of the 24 errors are `?`: the model explains before it answers, and its answer, cut after 30 tokens (`max_new_tokens`), doesn't name a category yet. Reading the answer fails too: in "rather than a network", the first category named isn't the model's choice, and two workstation tickets become `network`. Other errors do come from the model: the `answer` column shows it writing "access" in bold itself for a mailing list or a lost laptop.

In [5]:
from IPython.display import display

with pd.option_context("display.max_colwidth", None):
  display(errors(v1).join(v1.answer))

,ticket,expected,predicted,answer
0,My account is locked after several wrong passwords.,access,security,"Based on the description, the issue involves a user account being locked due to incorrect password attempts. This is a direct consequence of **security** measures ("
4,I no longer receive the two-factor authentication codes on my phone.,access,security,"Based on the issue described, the problem is related to the **security** of your account, specifically the two-factor authentication (2FA) mechanism."
6,I forgot my Windows password.,access,workstation,"**Workstation**\n\nThis ticket involves a user account on a Windows computer, which falls under the category of workstations."
9,No wifi in meeting room B.,network,access,**access**\n\n**Reasoning:**\nThe issue is about the availability of Wi-Fi (access) in a specific location (meeting room B).
10,The internet has been very slow on the whole floor since this morning.,network,?,"Based on the description of the issue, the problem is related to the speed and performance of the internet connection, which is a core component of the **"
15,"The intranet site is down, for everyone.",network,?,"Based on the description ""The intranet site is down, for everyone,"" the issue is a failure of the internal web service. This falls under"
18,How do I share my calendar with my assistant?,email,access,"Based on the nature of the request, this ticket should be categorized under **access**.\n\nSharing a calendar is a permission and access control task. It"
20,I'd like a mailing list for the project team.,email,access,"**access**\n\n**Reasoning:**\nWhile the request involves a project team, the specific tool requested is a **mailing list**, which is"
21,A 30 MB attachment is rejected when sending.,email,?,"Based on the description ""A 30 MB attachment is rejected when sending,"" this issue typically stems from file size limits, upload queue congestion, or"
22,My out-of-office replies aren't sent.,email,access,"Based on the nature of the issue, this ticket should be categorized under **access**.\n\n**Reasoning:**\nOut-of-office (OOO)"


## Version 2: defining the categories and the format

The model doesn't know our conventions: where does a VPN password go, or a phishing email? We define each category, with the priority rules, and enforce the answer's format.

In [6]:
PROMPT_V2 = """You sort IT support tickets. Categories:
- access: accounts, passwords (including the VPN's or the mailbox's), rights, two-factor authentication
- network: VPN, wifi, internet connection, slow or dropped network, unreachable internal sites
- email: emails, calendar, mailing lists, attachments
- workstation: the workstation's hardware and software, printers
- security: suspicious or fraudulent email, virus, lost or stolen equipment, suspicious login, data leak

Rule: anything about a threat or a security incident belongs to "security", even if it goes through an email or an account.

Answer with the category name only, in lowercase.

Ticket: {ticket}"""

v2 = evaluate(PROMPT_V2)
errors(v2)

Accuracy: 82% (33/40)


,ticket,expected,predicted
32,"An email from ""IT"" asks me to confirm my passw...",security,access
33,I clicked a link in an email and a page asked ...,security,access
35,I lost my work laptop on the train.,security,workstation
36,Pop-up ads open by themselves on my computer.,security,workstation
37,Someone logged into my account from abroad las...,security,access
38,A supplier asks by email to change their bank ...,security,email
39,I sent a customer file to an outside address b...,security,email


## Version 3: adding examples

We add a few solved examples, picked among the hard cases, but **not in the test set**: otherwise we would measure the model's memory, not its ability to generalize.

In [7]:
EXAMPLES = """Examples:
Ticket: My VPN access is locked after three wrong passwords. → access
Ticket: Screen sharing freezes during remote meetings with the branch. → network
Ticket: I can't find my old archived emails. → email
Ticket: My wireless mouse stopped working. → workstation
Ticket: A text message in the name of the company's bank asks for my credentials. → security

"""

PROMPT_V3 = PROMPT_V2.replace("Ticket: {ticket}", EXAMPLES + "Ticket: {ticket}")

v3 = evaluate(PROMPT_V3)
errors(v3)

Accuracy: 90% (36/40)


,ticket,expected,predicted
32,"An email from ""IT"" asks me to confirm my passw...",security,access
35,I lost my work laptop on the train.,security,?
38,A supplier asks by email to change their bank ...,security,email
39,I sent a customer file to an outside address b...,security,email


## Comparing the versions

Right answers per category, for each version:

In [8]:
pd.DataFrame(
  {
    version: (results.predicted == results.expected).groupby(results.expected).sum()
    for version, results in {"v1": v1, "v2": v2, "v3": v3}.items()
  }
)

,v1,v2,v3
expected,,,
access,5,8,8
email,3,8,8
network,5,8,8
security,1,1,4
workstation,2,8,8


Each version fixes some errors and may create others. In the run above, v2's definitions bring access, email, network and workstation to 8 right answers out of 8, but not security, still at 1: despite the rule, the model files a security incident under the channel it goes through, an account, an email or the computer itself. v3's examples bring security from 1 to 4 without breaking anything elsewhere: its 90% hides a category the model gets right only half the time. Hence measuring category by category, not only overall.

The best version's confusion matrix: the expected category in rows, the predicted one in columns.

In [9]:
pd.crosstab(v3.expected, v3.predicted)

predicted,?,access,email,network,security,workstation
expected,,,,,,
access,0,8,0,0,0,0
email,0,0,8,0,0,0
network,0,0,0,8,0,0
security,1,1,2,0,4,0
workstation,0,0,0,0,0,8


With 40 tickets, one ticket is worth 2.5 points: a 5-point gap between two versions may be chance. A real test set rather counts a hundred cases, taken from real tickets.

## Your turn

Write your own version of the prompt and measure it. The cell starts from v3: change `MY_PROMPT` to fix its weakest category, security (4 out of 8 above: a supplier changing their bank details by email and a customer file sent outside end up under "email"), for instance with one more rule or example, keeping `{ticket}` where the ticket goes. Without a GPU, allow about 6 minutes per try (a rough estimate).

In [10]:
# To modify, for instance:
# MY_PROMPT = PROMPT_V3.replace("Answer with", "Rule: …\n\nAnswer with")
MY_PROMPT = PROMPT_V3

mine = evaluate(MY_PROMPT)
errors(mine)

Accuracy: 90% (36/40)


,ticket,expected,predicted
32,"An email from ""IT"" asks me to confirm my passw...",security,access
35,I lost my work laptop on the train.,security,?
38,A supplier asks by email to change their bank ...,security,email
39,I sent a customer file to an outside address b...,security,email


## Key takeaways

- A prompt is judged on a labeled test set, not on a few tries
- Defining the categories and our conventions, enforcing the format, giving examples: each addition can be measured
- The prompt's examples must not come from the test set
- This test set is replayed on every change of prompt or model